---
layout: base
title: Hallpass
permalink: /bathroom/hallpass
menu: nav/toolkits/bathroom/menu.html
toc: false
---

{% include nav/toolkits/bathroom/menu.html %}

<div class="container">
    <div class="components">
        <!-- Resized Image -->
        <img src="https://github.com/user-attachments/assets/c350ab11-f560-4c76-b1f3-8bfbce017f77" alt="Teacher" style="max-width: 300px; height: auto; display: block; margin: 20px auto;">

        <p>Good job not using any paper, <span id="studentNameDisplay"></span>! You're saving both the environment and your seed!</p>

        <button type="button" onclick="window.approveStudent()">I'm done</button>
    </div>
</div>

<script type="module">
    import {javaURI, fetchOptions} from "{{site.baseurl}}/assets/js/api/config.js";

    // Backend API endpoints (No hardcoded localhost)
    const queueURI = javaURI + "/api/queue";
    const approvalURI = javaURI + "/api/approval";
    const tinkleURI = javaURI + "/api/tinkle";
    const personURI = javaURI + "/api/person";

    const getUrl = queueURI + "/all";
    const addUrl = queueURI + "/add";
    const removeUrl = queueURI + "/remove";
    const approvalUrl = approvalURI + "/approveRequest";
    const timeInUrl = (studentName) => `${tinkleURI}/timeIn/${studentName}`;

    const postOptions = {
        ...fetchOptions,
        method: "POST",
    };
    const deleteOptions = {
        ...fetchOptions,
        method: "DELETE",
    };

    // Get teacher name and student name from local storage or variables
    const teacherName = "jm1021@gmail.com"; // Replace with dynamic teacher name if needed

    window.addEventListener('load', () => {
        fetchUser();
    });


    async function fetchUser() {
        const response = await fetch(`${personURI}/get`, {
            method: "GET",
            cache: "no-cache",
            credentials: "include",
            headers: { 
                "Content-Type": "application/json",
                "X-Origin": "client"
            }
        });

        if (response.ok) {
            const userInfo = await response.json();
            window.studentName = userInfo.name;
            document.getElementById("studentNameDisplay").textContent = window.studentName;
            console.log("Person: ", userInfo);
        }
    }

    async function getTimeIn(studentName) {
        const response = await fetch(timeInUrl(studentName));

        if (response.ok) {
            const tinkleData = await response.text(); // Ensure text response
            console.log(`Retrieved timeIn: ${tinkleData}`);
            return tinkleData;
        } else {
            console.warn("Could not retrieve timeIn for", studentName);
            return null;
        }
    }

  async function removeFromQueue(studentName, teacherEmail) {        
    try {
        const requestBody = JSON.stringify({ teacherEmail, studentName });

        const response = await fetch(removeUrl, {
            method: "DELETE",  // Ensure it's DELETE, not POST
            headers: { "Content-Type": "application/json" },
            body: JSON.stringify({ teacherEmail, studentName })
        });

        const responseData = await response.text();
        if (response.ok) {
            console.log(`Successfully removed ${studentName} from the queue.`);
        } else {
            console.warn(`Failed to remove ${studentName} from queue.`);
        }
    } catch (error) {
        console.error("Error during removeFromQueue:", error);
    }
}


    // Function to approve student who is first in line
    window.approveStudent = async function() {
        const timeIn = await getTimeIn(window.studentName);
    
        if (!timeIn || timeIn.trim() === "") {
            console.log("Could not retrieve timeIn. Approval process halted.");
            return;
        }

        const now = new Date();
        const hours = now.getHours();
        const minutes = now.getMinutes().toString().padStart(2, "0");
        const seconds = now.getSeconds().toString().padStart(2, "0");
        const timeOut = `${hours}:${minutes}:${seconds}`;
        
        // alert(`TimeIn: ${timeIn}, TimeOut: ${timeOut}`);

        let timeSpentMinutes = 0;

        if (timeIn) {
            const [inHours, inMinutes, inSeconds] = timeIn.split(":").map(Number);
            const [outHours, outMinutes, outSeconds] = timeOut.split(":").map(Number);

            // Convert both times to total seconds since midnight
            const totalSecondsIn = inHours * 3600 + inMinutes * 60 + inSeconds;
            const totalSecondsOut = outHours * 3600 + outMinutes * 60 + outSeconds;

            // Calculate the difference in seconds
            const timeSpentSeconds = totalSecondsOut - totalSecondsIn;

            // Convert to minutes
            timeSpentMinutes = (timeSpentSeconds / 60).toFixed(2);
        }

        const timeInOut = `${timeIn}-${timeOut}`;
        const times = {
            studentEmail: window.studentName,
            timeIn: timeInOut,
            averageDuration: timeSpentMinutes,
        };

        const queuer = {
            teacherEmail: teacherName,
            studentName: window.studentName,
            uri: javaURI,
        };

        // Send time data
        fetch(`${tinkleURI}/add`, {
            ...postOptions,
            body: JSON.stringify(times),
        })
        .then(response => {
            if (response.ok) {
                console.log("Time added to database");
            } else {
                console.warn("Failed to add time to database.");
            }
        })
        .catch(error => console.error("Error adding time to database:", error));

        // Remove student from queue (now passing teacherEmail)
        await removeFromQueue(window.studentName, teacherName);

        window.location.href = "{{site.baseurl}}/profile";
    }
</script>